# 04_vjepa_plus_v1_jepa.ipynb – Visual Features to a Synthetic Descriptor Task

This notebook uses a frozen ImageNet-1K ResNet-18 feature extractor from `timm`. It is not a V-JEPA model.

## What this local analogue measures

1. Extract CIFAR-10 image features with the pinned `timm/resnet18.a1_in1k` weights.
2. Build two synthetic targets from image brightness and edge strength. These are not captions or text embeddings.
3. Train a small projector on a fixed training split.
4. Report cosine alignment and Recall@1 on held-out images.

This is a small synthetic descriptor task. It does not reproduce VL-JEPA, text retrieval, or a paper benchmark. The weight source is Apache-2.0; the CIFAR-10 dataset uses its own terms.

[Notebook 11](11_vl_jepa.ipynb) advances the toy lesson with real CIFAR-10 class-prompt embeddings and a linear-classifier baseline. It is still a supervised analogue, not a VL-JEPA reproduction.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from jepa_tutorial.storage import CHECKPOINTS_DIR, DATA_DIR

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import timm
from tqdm.auto import tqdm

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [ ]:
# Fixed CIFAR-10 subset and train/test split
DATA_SEED = 42
MODEL_SEED = 42
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

trainset = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True, transform=transform)
n_samples = 1200
subset_rng = np.random.default_rng(DATA_SEED)
indices = subset_rng.choice(len(trainset), n_samples, replace=False)
split_rng = np.random.default_rng(DATA_SEED + 1)
split_order = split_rng.permutation(n_samples)
train_indices = split_order[:960]
test_indices = split_order[960:]

class IndexedSubset(Dataset):
    def __init__(self, base, idx):
        self.base = base
        self.idx = list(idx)
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        x, y = self.base[self.idx[i]]
        return x, y

subset = IndexedSubset(trainset, indices)
loader = DataLoader(
    subset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
print(f"subset size: {len(subset)} | train: {len(train_indices)} | held-out: {len(test_indices)}")
print(f"data seed: {DATA_SEED} | model seed: {MODEL_SEED}")

In [ ]:
# Frozen ImageNet-1K ResNet-18 features (not V-JEPA features)
WEIGHT_REVISION = "491b427b45c94c7fb0e78b5474cc919aff584bbf"
encoder = timm.create_model(
    "resnet18.a1_in1k",
    pretrained=True,
    num_classes=0,
    pretrained_cfg_overlay={
        "hf_hub_id": f"timm/resnet18.a1_in1k@{WEIGHT_REVISION}"
    },
).to(device)
encoder.eval()

all_feats = []
all_labels = []
with torch.no_grad():
    for x, y in tqdm(loader, desc="Extract visual features"):
        x = x.to(device)
        z = encoder(x)
        z = F.normalize(z, dim=-1)
        all_feats.append(z.cpu())
        all_labels.append(y)

visual_feats = torch.cat(all_feats, dim=0)
labels = torch.cat(all_labels, dim=0)
print("visual_feats:", tuple(visual_feats.shape))
print("weight source: timm/resnet18.a1_in1k")
print("weight revision:", WEIGHT_REVISION)
print("weight license: Apache-2.0")

In [ ]:
# Synthetic brightness and edge-strength descriptors; these are not text targets.
def build_descriptors(images_batch):
    # Invert ImageNet channel normalization before calculating image statistics.
    mean = torch.tensor([0.485, 0.456, 0.406], device=images_batch.device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=images_batch.device).view(1, 3, 1, 1)
    imgs = (images_batch * std + mean).clamp(0, 1)
    gray = imgs.mean(dim=1, keepdim=True)
    gx = gray[:, :, :, 1:] - gray[:, :, :, :-1]
    gy = gray[:, :, 1:, :] - gray[:, :, :-1, :]
    edge_strength = (gx.abs().mean(dim=(1, 2, 3)) + gy.abs().mean(dim=(1, 2, 3))) / 2.0
    brightness = gray.mean(dim=(1, 2, 3))
    return torch.stack([brightness, edge_strength], dim=1)

descriptor_list = []
for x, _ in loader:
    descriptor_list.append(build_descriptors(x))
descriptors = torch.cat(descriptor_list, dim=0)
print("synthetic descriptors:", tuple(descriptors.shape))

In [ ]:
# Fit the projector and descriptor normalization on training images only.
target_mean = descriptors[train_indices].mean(dim=0, keepdim=True)
target_std = descriptors[train_indices].std(dim=0, keepdim=True, unbiased=False).clamp_min(1e-6)
standardized_targets = (descriptors - target_mean) / target_std

in_dim = visual_feats.shape[1]
target_dim = descriptors.shape[1]
projector = nn.Sequential(
    nn.Linear(in_dim, 256),
    nn.ReLU(),
    nn.Linear(256, target_dim),
).to(device)

opt = torch.optim.Adam(projector.parameters(), lr=2e-3)
v_train = visual_feats[train_indices].to(device)
t_train = standardized_targets[train_indices].to(device)

for epoch in range(80):
    pred = F.normalize(projector(v_train), dim=-1)
    tgt = F.normalize(t_train, dim=-1)
    loss = 1 - (pred * tgt).sum(dim=1).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()

    if epoch % 10 == 0 or epoch == 79:
        print(f"epoch {epoch:02d} | train cosine loss {loss.item():.4f}")

In [ ]:
# Held-out alignment and retrieval against synthetic descriptors
v_test = visual_feats[test_indices].to(device)
t_test = standardized_targets[test_indices].to(device)
with torch.no_grad():
    pred = F.normalize(projector(v_test), dim=-1)
    tgt = F.normalize(t_test, dim=-1)
    cosine_alignment = F.cosine_similarity(pred, tgt, dim=-1).mean().item()
    similarity = pred @ tgt.T
    nearest = similarity.argmax(dim=1)
    recall_at_1 = (nearest == torch.arange(len(nearest), device=device)).float().mean().item()

print(f"Held-out cosine alignment: {cosine_alignment:.3f}")
print(f"Held-out descriptor Recall@1: {recall_at_1:.3f}")
print(f"Held-out examples: {len(test_indices)}")

# Save the local analogue checkpoint outside source control.
ckpt_dir = CHECKPOINTS_DIR
ckpt_dir.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "projector": projector.state_dict(),
        "in_dim": in_dim,
        "target_dim": target_dim,
        "data_seed": DATA_SEED,
        "model_seed": MODEL_SEED,
        "weight_revision": WEIGHT_REVISION,
        "train_indices": train_indices,
        "test_indices": test_indices,
        "target_mean": target_mean,
        "target_std": target_std,
    },
    ckpt_dir / "vl_jepa_toy_projector.pt",
)
print("saved:", ckpt_dir / "vl_jepa_toy_projector.pt")

## Exercises

1. Run [Notebook 11](11_vl_jepa.ipynb) to replace synthetic targets with embeddings of real CIFAR-10 class prompts; compare its baseline and read its evidence limits.
2. Replace class prompts with human-written captions from a dataset whose terms and splits are documented, then measure retrieval.
3. Try contrastive InfoNCE loss instead of direct cosine regression and compare stability.
